# Clustering of Modal Construction Collexemes

This notebook compares two clustering approaches for the infinitives associated with an Italian modal construction:

1. collocation-based clustering, based on context windows around the modal + infinitive construction extracted from Sketch Engine concordance files;
2. embedding-based clustering, based on pretrained Italian fastText vectors.

The goal is to compare distributional patterns derived from corpus data and tightly bound to a specific modal construction (i.e. collocational patterns extracted from modal + infinitive contexts) with general semantic similarity captured by embeddings in a distributional vector space.

## Setup

In [ ]:
import re
import string
import glob
import math
import json
import csv
from pathlib import Path
from collections import Counter
from itertools import chain

import numpy as np
import pandas as pd
from tqdm import tqdm

from scipy.sparse import csr_matrix
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform
from sklearn.metrics.pairwise import cosine_distances

import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, FormatStrFormatter

In [ ]:
!pip install https://github.com/explosion/spacy-models/releases/download/it_core_news_sm-3.7.0/it_core_news_sm-3.7.0-py3-none-any.whl

import spacy

NLP = spacy.load("it_core_news_sm", disable=["parser", "ner", "textcat"])

In [ ]:
RAW_DIR = Path("data")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(42)

W = 5
REQUIRE_FULL = True
MIN_PER_SIDE = 3

USE_DF_PRUNE = True
DF_MIN_VERBS = 3
DF_MAX_SHARE = 0.60

## Text preprocessing

In [ ]:
STOP = NLP.Defaults.stop_words.union({
    "«", "»", "“", "”", "–", "—", "…", ".", ",", ";", ":", "!",
    "?", "(", ")", "[", "]", "{", "}", "\"", "'", "``", "''", "-", "_"
})

KEEP = {
    "non", "mai", "si", "più", "per", "senza", "entro", "forse",
    "probabilmente", "evidentemente", "sembra", "pare"
}

def spacy_tokenize_and_lemmatize(text):
    """Tokenize and lemmatize text using spaCy."""
    doc = NLP(text)
    lemmas = []
    for token in doc:
        if token.is_punct or token.is_space:
            continue
        lemma = token.lemma_.lower().strip()
        lemmas.append(lemma if lemma else token.text.lower())
    return lemmas


RE_HAS_DIGIT = re.compile(r"\d")
RE_FILE_EXT = re.compile(r"\.(doc|pdf|jpg|zip|it)$")
RE_URL_LIKE = re.compile(r"^(\/|http|www|\[|<|>)")

def clean_after_window(tokens):
    """Clean context-window tokens after lemmatization."""
    cleaned = []
    for t in tokens:
        t = t.strip()

        if t in KEEP:
            cleaned.append(t)
            continue

        if t in STOP:
            continue
        if RE_HAS_DIGIT.search(t):
            continue
        if len(t) <= 2:
            continue
        if RE_FILE_EXT.search(t):
            continue
        if RE_URL_LIKE.match(t):
            continue

        t = t.strip(string.punctuation + "«»“”–—…")
        if not t or t in STOP:
            continue

        cleaned.append(t)

    return cleaned

In [ ]:
TAG_L = re.compile(r".*<s>", flags=re.DOTALL)
TAG_R = re.compile(r"</s>.*", flags=re.DOTALL)

def clipL(s):
    return TAG_L.sub("", str(s))

def clipR(s):
    return TAG_R.sub("", str(s))

## Modal selection

Select the modal verb to process by setting `USE_MODAL`. The corresponding forms and filename pattern are defined below.

In [ ]:
USE_MODAL = "volere"

In [ ]:
if USE_MODAL == "dovere":
    MODAL_FORMS = (
        r"(?:devo|devi|deve|dobbiamo|dovete|devono|debba|dobbiate|debbano|"
        r"dovevo|dovevi|doveva|dovevamo|dovevate|dovevano|"
        r"dovr[àa]|dovrò|dovrai|dovremo|dovrete|dovranno|dovrei|dovresti|dovrebbe|"
        r"dovremmo|dovreste|dovrebbero|dovere|dovuto|dovessi|dovesse|dovessimo|"
        r"doveste|dovessero|dovendo)"
    )
    FILENAME_RE = re.compile(r"dovere[ _-]*inf[ _-]*-?([^.]+)\.csv$", flags=re.IGNORECASE)

elif USE_MODAL == "potere":
    MODAL_FORMS = (
        r"(?:posso|puoi|può|possiamo|potete|possono|"
        r"potevo|potevi|poteva|potevamo|potevate|potevano|potetti|potei|potesti|"
        r"potette|poté|potemmo|poteste|potettero|poterono|potr[òa]|potrai|"
        r"potremo|potrete|potranno|potrei|potresti|potrebbe|potremmo|potreste|"
        r"potrebbero|possa|possiate|possano|potere|potuto|potessi|potesse|"
        r"potessimo|poteste|potessero|potendo)"
    )
    FILENAME_RE = re.compile(r"potere[ _-]*inf[ _-]*-?([^.]+)\.csv$", flags=re.IGNORECASE)

elif USE_MODAL == "volere":
    MODAL_FORMS = (
        r"(?:voglio|vuoi|vuole|vogliamo|volete|vogliono|"
        r"volevo|volevi|voleva|volevamo|volevate|volevano|volli|volesti|volle|"
        r"volemmo|voleste|vollero|vorr[òa]|vorrai|vorremo|vorrete|vorranno|"
        r"vorrei|vorresti|vorrebbe|vorremmo|vorreste|vorrebbero|voglia|vogliate|"
        r"vogliano|volere|voluto|volessi|volesse|volessimo|voleste|volessero|volendo)"
    )
    FILENAME_RE = re.compile(r"volere[ _-]*inf[ _-]*-?([^.]+)\.csv$", flags=re.IGNORECASE)

else:
    raise ValueError("USE_MODAL must be one of: 'dovere', 'potere', 'volere'.")

RE_MODAL = re.compile(rf"\b{MODAL_FORMS}\b", re.IGNORECASE)

In [ ]:
def lemma_from_filename(fp: Path):
    """Extract infinitive lemma from filename."""
    name = fp.name
    m = FILENAME_RE.search(name)

    if m:
        return m.group(1).strip()

    base = Path(name).stem
    return re.split(r"[-_ ]+", base)[-1]

## Reading Sketch Engine exports

In [ ]:
def read_sketchengine_kwic_csv(fp: Path, max_lines=10):
    """
    Read Sketch Engine KWIC CSV export.

    Expected output columns:
    Reference, Left, KWIC, Right.
    """
    with open(fp, "r", encoding="utf-8", errors="ignore") as f:
        lines = []
        for _ in range(max_lines):
            try:
                lines.append(next(f))
            except StopIteration:
                break

    norm = [l.strip().replace("\ufeff", "").lower() for l in lines]

    header_idx = None
    header_sep = None
    header_text = None

    for i, ln in enumerate(norm):
        if ("reference" in ln) and (("kwic" in ln) or ("node" in ln)) and ("left" in ln) and ("right" in ln):
            header_idx = i
            seps = [(",", ln.count(",")), (";", ln.count(";")), ("\t", ln.count("\t"))]
            seps.sort(key=lambda x: x[1], reverse=True)
            header_sep = seps[0][0] if seps[0][1] >= 3 else ","
            header_text = lines[i].strip().replace("\ufeff", "")
            break

    if header_idx is None:
        preview = "\n".join(lines[:8])
        raise ValueError(f"Could not find header in {fp.name}. First lines:\n{preview}")

    df = pd.read_csv(
        fp,
        engine="python",
        sep=header_sep,
        skiprows=header_idx,
        header=0,
        on_bad_lines="skip",
        dtype=str,
        quoting=csv.QUOTE_MINIMAL,
    )

    df.rename(columns={c: c.strip() for c in df.columns}, inplace=True)

    if "KWIC" not in df.columns:
        if "Kwic" in df.columns:
            df.rename(columns={"Kwic": "KWIC"}, inplace=True)
        elif "Node" in df.columns:
            df.rename(columns={"Node": "KWIC"}, inplace=True)

    if "Left context" in df.columns:
        df.rename(columns={"Left context": "Left"}, inplace=True)

    if "Right context" in df.columns:
        df.rename(columns={"Right context": "Right"}, inplace=True)

    expected = ["Reference", "Left", "KWIC", "Right"]
    missing = [c for c in expected if c not in df.columns]

    if missing:
        raise ValueError(
            f"{fp.name}: missing columns {missing}. "
            f"Got: {list(df.columns)} | sep='{header_sep}' | header='{header_text}'"
        )

    return df[expected].dropna(how="all")

## Building context windows

In [ ]:
ENFORCE_KWIC_HIT = True

def build_windows(fp: Path, W=5, enforce_kwic_hit=ENFORCE_KWIC_HIT):
    """Build cleaned context windows around modal–infinitive constructions."""
    lemma = lemma_from_filename(fp)
    df = read_sketchengine_kwic_csv(fp)

    rows = []

    for _, r in df.iterrows():
        L_lemmas = spacy_tokenize_and_lemmatize(clipL(r["Left"]))
        K_lemmas = spacy_tokenize_and_lemmatize(r["KWIC"])
        R_lemmas = spacy_tokenize_and_lemmatize(clipR(r["Right"]))

        if enforce_kwic_hit and lemma not in (L_lemmas + K_lemmas + R_lemmas):
            continue

        left_ctx = L_lemmas[-W:]
        right_ctx = R_lemmas[:W]

        ok = (
            len(left_ctx) >= (W if REQUIRE_FULL else MIN_PER_SIDE)
            and len(right_ctx) >= (W if REQUIRE_FULL else MIN_PER_SIDE)
        )

        if not ok:
            continue

        drop = {lemma}
        win = [
            t for t in (left_ctx + right_ctx)
            if t not in drop and not RE_MODAL.fullmatch(t)
        ]

        win = clean_after_window(win)

        if not win:
            continue

        rows.append({
            "verb": lemma,
            "types": win,
            "reference": r["Reference"]
        })

    return rows

In [ ]:
all_files = sorted(list(RAW_DIR.glob("*.csv")))
print(f"Found {len(all_files)} files.")

all_rows = []

for fp in tqdm(all_files, desc="Processing files"):
    try:
        rows = build_windows(fp, W=W, enforce_kwic_hit=ENFORCE_KWIC_HIT)
        all_rows.extend(rows)
    except Exception as e:
        print(f"Error processing {fp.name}: {e}")

df_windows = pd.DataFrame(all_rows)
print(f"Built {len(df_windows)} window rows from all files.")

## Vocabulary construction and pruning

In [ ]:
type_freq = Counter()
verb_doc_freq = Counter()  # how many verbs a type co-occurs with

for _, row in df_windows.iterrows():
    types = row["types"]
    verb = row["verb"]
    type_freq.update(types)
    verb_doc_freq.update(set(types))

print(f"Total unique context types: {len(type_freq)}")

EXCLUDED_TYPES = {'a.a', 'a.m', 'a.s', 'a.s.l', 'aams'}

if USE_DF_PRUNE:
    total_verbs = len(set(df_windows["verb"]))
    max_share = total_verbs * DF_MAX_SHARE
    kept_types = {
        t for t, df in verb_doc_freq.items()
        if df >= DF_MIN_VERBS and df <= max_share and t not in EXCLUDED_TYPES
    }
else:
    kept_types = {t for t in type_freq.keys() if t not in EXCLUDED_TYPES}

print(f"Vocabulary size after DF pruning: {len(kept_types)}")

def filter_types(types):
    return [t for t in types if t in kept_types]

df_windows["types_filtered"] = df_windows["types"].apply(filter_types)
df_windows = df_windows[df_windows["types_filtered"].map(len) > 0].reset_index(drop=True)

## PPMI matrix construction

In [ ]:
type2idx = {t: i for i, t in enumerate(sorted(kept_types))}
verb2idx = {v: i for i, v in enumerate(sorted(df_windows["verb"].unique()))}

rows_idx = []
cols_idx = []
data_vals = []

print("Vectorizing term frequencies...")

for _, row in tqdm(df_windows.iterrows(), total=len(df_windows)):
    verb_i = verb2idx[row["verb"]]

    for t in row["types_filtered"]:
        t_i = type2idx[t]
        rows_idx.append(verb_i)
        cols_idx.append(t_i)
        data_vals.append(1)

X_counts = csr_matrix(
    (data_vals, (rows_idx, cols_idx)),
    shape=(len(verb2idx), len(type2idx)),
    dtype=np.int32
)

X_counts.sum_duplicates()

print(f"Sparse count matrix shape: {X_counts.shape}")
print(f"Number of nonzero entries: {X_counts.nnz}")

print("Computing PPMI matrix...")

N = X_counts.sum()

row_sums = np.array(X_counts.sum(axis=1)).flatten()
col_sums = np.array(X_counts.sum(axis=0)).flatten()

X_coo = X_counts.tocoo()

pmi_data = []

for i, j, v in zip(X_coo.row, X_coo.col, X_coo.data):
    p_xy = v / N
    p_x = row_sums[i] / N
    p_y = col_sums[j] / N

    pmi = np.log2(p_xy / (p_x * p_y)) if p_xy > 0 else 0
    pmi_data.append(max(pmi, 0))

X_ppmi = csr_matrix(
    (pmi_data, (X_coo.row, X_coo.col)),
    shape=X_counts.shape
)

print(f"Sparse PPMI matrix shape: {X_ppmi.shape}")
print(f"Number of nonzero entries in PPMI: {X_ppmi.nnz}")

## Collocation-based clustering

In [ ]:
print("Computing cosine distances...")
dist = cosine_distances(X_ppmi)

condensed_dist = squareform(dist)

print("Clustering...")
linkage_matrix = linkage(condensed_dist, method="complete")

max_h = linkage_matrix[:, 2].max()
color_thr = 0.5 * max_h

plt.figure(figsize=(12, 5))

dendrogram(
    linkage_matrix,
    labels=list(verb2idx.keys()),
    orientation="top",
    distance_sort="ascending",
    leaf_rotation=90,
    leaf_font_size=10,
    color_threshold=color_thr,
    above_threshold_color="grey",
)

plt.xlabel("Verbs")
plt.ylabel("Cosine Distance")

ax = plt.gca()
ax.yaxis.set_major_locator(MultipleLocator(0.1))
ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
ax.set_ylim(0, max_h * 1.02)

plt.tight_layout()

out_path = OUT_DIR / "dendrogram_vol_coll.png"
plt.savefig(out_path, dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
vocab = sorted(type2idx.keys())

print(f"Number of collocate types in vocabulary: {len(vocab)}")
print(vocab)

## Embedding-based clustering with fastText

In [ ]:
!pip install fasttext

import fasttext
import fasttext.util

In [ ]:
fasttext.util.download_model("it", if_exists="ignore")
ft = fasttext.load_model("cc.it.300.bin")

print("Embedding dimension:", ft.get_dimension())

## Collexeme list for embedding-based clustering

The following list contains the top 40 collexemes for the selected modal construction.

In [ ]:
COLLEXEMES_BY_MODAL = {
    "dovere": [
        "accontentare", "affrontare", "ammettere", "arrendere", "aspettare",
        "attendere", "attenere", "avvenire", "cambiare", "compilare",
        "confessare", "confrontare", "dire", "fare", "fronteggiare",
        "imparare", "intervenire", "lottare", "pagare", "pervenire",
        "possedere", "preoccupare", "procedere", "provvedere", "rassegnare",
        "recare", "ricorrere", "ricredere", "rifare", "rinunciare",
        "rispettare", "rispondere", "sborsare", "smettere", "sopportare",
        "sottostare", "subire", "superare", "tenere", "vergognare"
    ],

    "potere": [
        "usufruire", "ammirare", "accedere", "contare", "variare",
        "beneficiare", "causare", "aiutare", "esimere", "vantare",
        "gustare", "prescindere", "immaginare", "godere", "degustare",
        "fruire", "sbizzarrire", "constatare", "capitare", "fregiare",
        "influire", "optare", "avvalere", "obiettare", "interferire",
        "scaricare", "coesistere", "visionare", "desumere", "derogare",
        "rivalere", "recedere", "compromettere", "consultare", "nuocere",
        "detrarre", "dedurre", "contattare", "provocare", "insorgere"
    ],

    "volere": [
        "sapere", "dire", "fare", "ringraziare", "vedere", "dare",
        "provare", "chiedere", "ricordare", "approfondire", "sottolineare",
        "parlare", "capire", "mettere", "rinunciare", "condividere",
        "segnalare", "cambiare", "partecipare", "evitare", "spendere",
        "raccontare", "cimentare", "conoscere", "comprare", "acquistare",
        "tornare", "intraprendere", "imparare", "scopare", "esprimere",
        "soffermare", "aggiungere", "andare", "regalare", "continuare",
        "sentire", "vendicare", "mantenere", "vivere"
    ]
}

verbs = COLLEXEMES_BY_MODAL[USE_MODAL]

assert len(verbs) == 40

In [ ]:
embeddings = np.array([ft.get_word_vector(verb) for verb in verbs])

print("Embeddings shape:", embeddings.shape)

In [ ]:
distance_matrix = cosine_distances(embeddings)
condensed_dist = squareform(distance_matrix, checks=False)

In [ ]:
linked = linkage(condensed_dist, method='complete')

In [ ]:
max_h = linked[:, 2].max()
color_thr = 0 * max_h

plt.figure(figsize=(12, 5))

dendrogram(
    linked,
    labels=verbs,
    orientation="top",
    distance_sort="ascending",
    leaf_rotation=90,
    leaf_font_size=10,
    color_threshold=color_thr,
    above_threshold_color="grey",
)

plt.xlabel("Verbs")
plt.ylabel("Cosine Distance")
plt.tight_layout()

plt.savefig(OUT_DIR / "dendrogram_vol_emb.png", dpi=300, bbox_inches="tight")

plt.show()